# SIMPLIcity: EfficientNet-B0 Final Enhancement (V3 Streamlined)
### Canonical 224x224 Full-Canvas Cropping, Streamlined Linear Head, and End-to-End Fine-Tuning

This notebook trains and evaluates the optimized **EfficientNet-B0 (V3)** pipeline on the SIMPLIcity dataset (1,000 images, 10 classes) using **Dual NVIDIA Tesla T4 GPUs** (`torch.nn.DataParallel`).

---
### Forensic Diagnosis & V3 Systematic Fixes:
1. **Eliminated Letterbox Padding**: Replaced symmetric grey padding bars (which caused a 33.3% dead canvas diluting Global Average Pooling) with canonical ImageNet `224x224` `RandomResizedCrop(scale=(0.85, 1.0))` for training and deterministic `Resize(256) -> CenterCrop(224)` for testing.
2. **Streamlined Single-Layer Head**: Replaced the 330k parameter 2-layer projected head with a calibrated single-layer classifier (`nn.Dropout(p=0.25) -> nn.Linear(1280, 10)`), reducing head parameter footprint by 96% and preserving ImageNet semantic representations.
3. **End-to-End Differential Fine-Tuning**: Removed the 5-epoch frozen warm-up to prevent backbone starvation during Cosine LR decay; trains end-to-end with differential learning rate ($2.5 \times 10^{-5}$ backbone vs $2.5 \times 10^{-4}$ head).
4. **Calibrated Label Smoothing (0.04)**: Softened over-regularization from V2 (0.08) to allow cross-entropy loss to descend naturally below 0.38 while retaining generalization benefits.
5. **Strict Deterministic Single-Pass Evaluation**: Evaluates purely on unaugmented test images without synthetic TTA distortions.

## 1. Verify GPU Acceleration & Dual T4 Topology
Checks CUDA availability and device properties across both Tesla T4 GPUs.

In [ ]:
!nvidia-smi

import os

import torch

print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
count = torch.cuda.device_count()
print(f"GPU Count       : {count}")
for i in range(count):
    name = torch.cuda.get_device_name(i)
    props = torch.cuda.get_device_properties(i)
    mem = getattr(props, "total_memory", getattr(props, "total_mem", 0)) / (1024**3)
    print(f"  GPU {i}: {name} ({mem:.1f} GB VRAM)")
if count > 1:
    print(">> Dual GPU detected. DataParallel acceleration enabled across all cards <<")

## 2. Setup Environment & Sync Repository
Pulls the latest code updates from the GitHub repository.

In [ ]:
import sys
from pathlib import Path

# 1. Ensure working directory is project root
if os.path.exists("src/simplicity_cnn"):
    repo_root = Path.cwd()
elif os.path.exists("../src/simplicity_cnn"):
    %cd ..
    repo_root = Path.cwd()
elif os.path.exists("cnn-SIMPLIcity/src/simplicity_cnn"):
    %cd cnn-SIMPLIcity
    repo_root = Path.cwd()
elif Path("/kaggle/working/cnn-SIMPLIcity/src/simplicity_cnn").exists():
    %cd /kaggle/working/cnn-SIMPLIcity
    repo_root = Path.cwd()
else:
    print("Cloning latest repository from GitHub...")
    !git clone -b main https://github.com/Arfiadi/cnn-SIMPLIcity.git
    %cd cnn-SIMPLIcity
    repo_root = Path.cwd()

# 2. Pull latest updates from main
!git pull origin main

# 3. Configure sys.path, PYTHONPATH, and install package in editable mode
src_path = str((repo_root / "src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ["PYTHONPATH"] = f"{src_path}:{os.environ.get('PYTHONPATH', '')}"

%pip install -q -e . --no-deps
print(f"Active working directory: {repo_root}")
print(f"Configured sys.path and PYTHONPATH with: {src_path}")

## 3. Dataset Verification
Auto-detects and audits dataset split disjointness and file counts.

In [ ]:
!python scripts/audit_data.py

## 4. Inspect Final EfficientNet-B0 (V3) Configuration
Displays the streamlined parameters (224x224, full canvas, linear head, end-to-end training).

In [ ]:
from simplicity_cnn.config.schema import print_config_summary

cfg = print_config_summary("configs/efficientnet_b0.yaml")

## 5. Execute Training: Final EfficientNet-B0 (V3)
Trains EfficientNet-B0 with Streamlined Linear Head and Differential LR across Dual Tesla T4 GPUs.

In [ ]:
!python scripts/train.py --config configs/efficientnet_b0.yaml --multi-gpu --backbone-lr-ratio 0.1

## 6. Training Curves & Convergence Diagnostics
Plots loss convergence, accuracy progression, and confusion matrix.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

out_dir = Path("outputs/efficientnet_b0_v3_final")
summary_plot = out_dir / "gradient_descent_summary.png"
cm_plot = out_dir / "evaluation" / "confusion_matrix.png"

if summary_plot.exists():
    print("=== Training Summary Curves ===")
    display(Image(filename=str(summary_plot)))

if cm_plot.exists():
    print("=== Test Set Confusion Matrix ===")
    display(Image(filename=str(cm_plot)))

## 7. 3-Way Ablation Analysis (Baseline vs V2 Letterbox vs V3 Final)
Compares per-class F1 scores and overall accuracy across all 3 development iterations.

In [ ]:
import json
from pathlib import Path

import pandas as pd

report_file = Path("outputs/efficientnet_b0_v3_final/evaluation/classification_report.json")
if report_file.exists():
    with open(report_file, "r", encoding="utf-8") as f:
        report = json.load(f)

    classes = [
        "People",
        "Beach",
        "Building",
        "Bus",
        "Dinosaur",
        "Elephant",
        "Flower",
        "Horse",
        "Mountain",
        "Food",
    ]
    baseline_f1 = {
        "People": 0.9744,
        "Beach": 0.8889,
        "Building": 0.8889,
        "Bus": 1.0000,
        "Dinosaur": 1.0000,
        "Elephant": 1.0000,
        "Flower": 1.0000,
        "Horse": 1.0000,
        "Mountain": 0.9000,
        "Food": 1.0000,
    }
    v2_letterbox_f1 = {
        "People": 0.9756,
        "Beach": 0.8837,
        "Building": 0.8571,
        "Bus": 0.9756,
        "Dinosaur": 1.0000,
        "Elephant": 0.9756,
        "Flower": 1.0000,
        "Horse": 1.0000,
        "Mountain": 0.8718,
        "Food": 1.0000,
    }
    rows = []
    for c in classes:
        if c in report:
            prec = report[c]["precision"]
            rec = report[c]["recall"]
            f1_v3 = report[c]["f1-score"]
            f1_base = baseline_f1.get(c, 0.0)
            f1_v2 = v2_letterbox_f1.get(c, 0.0)
            delta_base = (f1_v3 - f1_base) * 100
            delta_v2 = (f1_v3 - f1_v2) * 100
            rows.append(
                {
                    "Class": c,
                    "Precision": f"{prec:.4f}",
                    "Recall": f"{rec:.4f}",
                    "Baseline F1": f"{f1_base:.4f}",
                    "V2 (Letterbox)": f"{f1_v2:.4f}",
                    "V3 (Final)": f"{f1_v3:.4f}",
                    "Δ vs Base": f"{delta_base:+.2f}%",
                    "Δ vs V2": f"{delta_v2:+.2f}%",
                }
            )
    df = pd.DataFrame(rows)
    print("=" * 95)
    print(" EFFICIENTNET-B0: 3-WAY ABLATION STUDY (BASELINE vs V2 LETTERBOX vs V3 FINAL)")
    print("=" * 95)
    print(df.to_string(index=False))
    print("-" * 95)
    acc = report.get("accuracy", 0) * 100
    macro_f1 = report.get("macro avg", {}).get("f1-score", 0)
    print(" [V1 Baseline]   Accuracy: 96.50% | Macro F1: 0.9652 | Errors: 7/200")
    print(" [V2 Letterbox]  Accuracy: 95.50% | Macro F1: 0.9539 | Errors: 9/200")
    print(f" [V3 Final]      Accuracy: {acc:.2f}% | Macro F1: {macro_f1:.4f}")
    print("=" * 95)
else:
    print(f"Evaluation report not found at {report_file}. Run training first.")

## 8. Misclassification Error Inspection (V3 Final)
Inspects remaining test errors with confidence scores against Baseline (7) and V2 (9).

In [ ]:
import json
from pathlib import Path

err_file = Path("outputs/efficientnet_b0_v3_final/evaluation/misclassified_samples.json")
if err_file.exists():
    with open(err_file, "r", encoding="utf-8") as f:
        errors = json.load(f)
    print(f"Total Misclassified Samples: {len(errors)} / 200 (Baseline: 7 errors | V2: 9 errors)")
    for i, err in enumerate(errors, 1):
        fn = err["filename"]
        t_lbl = err["true_label"]
        p_lbl = err["predicted_label"]
        c_val = err["confidence"]
        print(
            f"  [{i:02d}] File: {fn:<8} | True: {t_lbl:<10} -> Predicted: {p_lbl:<10} (Conf: {c_val:.4f})"
        )
else:
    print(f"Error report not found at {err_file}.")